In [0]:
catalog_name='products'

In [0]:
import requests
import pandas as pd

from pyspark.sql.functions import (
    current_timestamp,
    lit,
    col
)

url = "https://dummyjson.com/products"

response = requests.get(url)

response.raise_for_status()

data = response.json()

products = data["products"]

pdf = pd.json_normalize(products)

new_df = spark.createDataFrame(pdf)

In [0]:
new_products= new_df.select(
    col('id').cast('int').alias('product_id'),
    col('title'),
    col('description'),
    col('category'),
    col('price').cast('double'),
    col('discountPercentage').cast('double'),
    col('rating').cast('double'),
    col('stock').cast('int'),
    col('brand'),
    col('sku'),
    col('weight').cast('double'),
    col("warrantyInformation"),
    col("shippingInformation"),
    col('availabilityStatus'),
    col('returnPolicy'),
    col("minimumOrderQuantity").cast('int'),
    col('thumbnail'),
    #dim
    col("`dimensions.width`").cast('double').alias('dim_width'),
    col("`dimensions.height`").cast('double').alias('dim_height'),
    col("`dimensions.depth`").cast('double').alias('dim_depth'),
    #meta
    col("`meta.createdAt`").alias('created_at'),
    col("`meta.updatedAt`").alias('updated_at'),
    col("`meta.barcode`").alias('barcode'),
    col("`meta.qrCode`").alias('qr_code'),
)

In [0]:
from pyspark.sql.functions import (
    current_timestamp,
    lit
)

new_products = (
    new_df.withColumn(
        "ingested_at",
        current_timestamp()
    )
    .withColumn(
        "data_source",
        lit("dummyjson_api")
    )
)
new_products.printSchema()

root
 |-- id: long (nullable = true)
 |-- title: string (nullable = true)
 |-- description: string (nullable = true)
 |-- category: string (nullable = true)
 |-- price: double (nullable = true)
 |-- discountPercentage: double (nullable = true)
 |-- rating: double (nullable = true)
 |-- stock: long (nullable = true)
 |-- tags: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- brand: string (nullable = true)
 |-- sku: string (nullable = true)
 |-- weight: long (nullable = true)
 |-- warrantyInformation: string (nullable = true)
 |-- shippingInformation: string (nullable = true)
 |-- availabilityStatus: string (nullable = true)
 |-- reviews: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- comment: string (nullable = true)
 |    |    |-- date: string (nullable = true)
 |    |    |-- rating: long (nullable = true)
 |    |    |-- reviewerEmail: string (nullable = true)
 |    |    |-- reviewerName: string (nullable = true)
 |--

In [0]:
from delta.tables import DeltaTable

delta_table = DeltaTable.forName(
    spark,
     f"{catalog_name}.silver.silver_products"
)

In [0]:
from pyspark.sql.functions import col

# Transform new_products to match target schema
merge_df = new_products.withColumnRenamed("id", "product_id").select(
    "product_id", "title", "description", "category", "price",
    "discountPercentage", "rating", "stock", "brand", "sku",
    "weight", "warrantyInformation", "shippingInformation",
    "availabilityStatus", "returnPolicy", "minimumOrderQuantity",
    "thumbnail",
    col("`dimensions.width`").alias("dim_width"),
    col("`dimensions.height`").alias("dim_height"),
    col("`dimensions.depth`").alias("dim_depth"),
    col("`meta.createdAt`").alias("created_at"),
    col("`meta.updatedAt`").alias("updated_at"),
    col("`meta.barcode`").alias("barcode"),
    col("`meta.qrCode`").alias("qr_code"),
    "ingested_at", "data_source"
)

delta_table.alias("target").merge(
    merge_df.alias("source"),
    "target.product_id = source.product_id"
).whenMatchedUpdateAll() \
 .whenNotMatchedInsertAll() \
 .execute()

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

In [0]:
display(
    spark.sql("""
    SELECT *
    FROM products.silver.silver_products
    LIMIT 10
    """)
)

product_id,title,description,category,price,discountPercentage,rating,stock,brand,sku,weight,warrantyInformation,shippingInformation,availabilityStatus,returnPolicy,minimumOrderQuantity,thumbnail,dim_width,dim_height,dim_depth,created_at,updated_at,barcode,qr_code,ingested_at,data_source
19,Chicken Meat,"Fresh and tender chicken meat, suitable for various culinary preparations.",groceries,9.99,13.7,3.19,97,null,GRO-BRD-CHI-019,1.0,1 year warranty,Ships in 1 month,In Stock,7 days return policy,22,https://cdn.dummyjson.com/product-images/groceries/chicken-meat/thumbnail.webp,11.03,22.11,16.01,2025-04-30T09:41:02.053Z,2025-04-30T09:41:02.053Z,8829514594521,https://cdn.dummyjson.com/public/qr-code.png,2026-06-06T20:35:48.719Z,dummyjson_api
20,Cooking Oil,"Versatile cooking oil suitable for frying, sautéing, and various culinary applications.",groceries,4.99,9.33,4.8,10,null,GRO-BRD-COO-020,5.0,Lifetime warranty,Ships in 1-2 business days,In Stock,30 days return policy,46,https://cdn.dummyjson.com/product-images/groceries/cooking-oil/thumbnail.webp,19.95,27.54,24.86,2025-04-30T09:41:02.053Z,2025-04-30T09:41:02.053Z,4874727824518,https://cdn.dummyjson.com/public/qr-code.png,2026-06-06T20:35:48.719Z,dummyjson_api
21,Cucumber,"Crisp and hydrating cucumbers, ideal for salads, snacks, or as a refreshing side.",groceries,1.49,0.16,4.07,84,null,GRO-BRD-CUC-021,4.0,2 year warranty,Ships in 1-2 business days,In Stock,7 days return policy,2,https://cdn.dummyjson.com/product-images/groceries/cucumber/thumbnail.webp,12.8,28.38,21.34,2025-04-30T09:41:02.053Z,2025-04-30T09:41:02.053Z,5300066378225,https://cdn.dummyjson.com/public/qr-code.png,2026-06-06T20:35:48.719Z,dummyjson_api
22,Dog Food,Specially formulated dog food designed to provide essential nutrients for your canine companion.,groceries,10.99,10.27,4.55,71,null,GRO-BRD-FOO-022,10.0,No warranty,Ships in 1-2 business days,In Stock,60 days return policy,43,https://cdn.dummyjson.com/product-images/groceries/dog-food/thumbnail.webp,16.93,27.15,9.29,2025-04-30T09:41:02.053Z,2025-04-30T09:41:02.053Z,5906686116469,https://cdn.dummyjson.com/public/qr-code.png,2026-06-06T20:35:48.719Z,dummyjson_api
23,Eggs,"Fresh eggs, a versatile ingredient for baking, cooking, or breakfast.",groceries,2.99,11.05,2.53,9,null,GRO-BRD-EGG-023,2.0,1 week warranty,Ships in 1 week,In Stock,No return policy,32,https://cdn.dummyjson.com/product-images/groceries/eggs/thumbnail.webp,11.42,7.44,16.95,2025-04-30T09:41:02.053Z,2025-04-30T09:41:02.053Z,3478638588469,https://cdn.dummyjson.com/public/qr-code.png,2026-06-06T20:35:48.719Z,dummyjson_api
24,Fish Steak,"Quality fish steak, suitable for grilling, baking, or pan-searing.",groceries,14.99,4.23,3.78,74,null,GRO-BRD-FIS-024,6.0,1 month warranty,Ships in 3-5 business days,In Stock,60 days return policy,50,https://cdn.dummyjson.com/product-images/groceries/fish-steak/thumbnail.webp,14.95,26.31,11.27,2025-04-30T09:41:02.053Z,2025-04-30T09:41:02.053Z,9595036192098,https://cdn.dummyjson.com/public/qr-code.png,2026-06-06T20:35:48.719Z,dummyjson_api
25,Green Bell Pepper,"Fresh and vibrant green bell pepper, perfect for adding color and flavor to your dishes.",groceries,1.29,0.16,3.25,33,null,GRO-BRD-GRE-025,2.0,1 month warranty,Ships in 1 week,In Stock,30 days return policy,12,https://cdn.dummyjson.com/product-images/groceries/green-bell-pepper/thumbnail.webp,15.33,26.65,14.44,2025-04-30T09:41:02.053Z,2025-04-30T09:41:02.053Z,2365227493323,https://cdn.dummyjson.com/public/qr-code.png,2026-06-06T20:35:48.719Z,dummyjson_api
26,Green Chili Pepper,"Spicy green chili pepper, ideal for adding heat to your favorite recipes.",groceries,0.99,1.0,3.66,3,null,GRO-BRD-GRE-026,7.0,2 year warranty,Ships in 1 week,Low Stock,30 days return policy,39,https://cdn.dummyjson.com/product-images/groceries/green-chili-pepper/thumbnail.webp,15.38,18.12,19.92,2025-04-30T09:41:02.053Z,2025-04-30T09:41:02.053Z,9335000538563,https://cdn.dummyjson.com/public/qr-code.png,2026-06-06T20:35:48.719Z,dummyjson